# Fast Iterative Method (FIM) in CUDA

Implementazione passo-passo degli **Algoritmi 6 e 7** della tesi (cap. 3.3), con test finale su una **sorgente puntiforme** in mezzo omogeneo (cap. 4.1).

**Prima di tutto:** *Runtime → Cambia tipo di runtime → Acceleratore hardware: T4 GPU*.

## Controllo GPU
Da eseguire all'inizio di ogni sessione Colab: deve comparire una **Tesla T4**. Se non c'è: *Runtime → Cambia tipo di runtime → T4 GPU*.

In [ ]:
!nvidia-smi

: 

## Step 1 – Algoritmo 2 + Algoritmo 6 in versione sequenziale (CPU)

Obiettivo: capire la logica del FIM **senza** la complessità della GPU e ottenere una **soluzione di riferimento** per gli step CUDA.

- `solve_quadratic` → **Algoritmo 2** (eq. 3.1), marcata `__host__ __device__` per riusarla sulla GPU.
- `solve_node` → la *g(x)* della tesi: legge i 4 vicini (INF fuori dal dominio) e chiama l'Algoritmo 2.
- `test_neighbors` → il "test dei nodi adiacenti" dell'Algoritmo 6.
- `fim_cpu` → il ciclo `while A ≠ ∅` dell'**Algoritmo 6**.
- `relative_error` → eq. (4.2), confronto con U = √(x²+y²).

Test: dominio [−1,1]², sorgente al centro, F = 1, N = 51, 101, 201, 401.
Per ora solo stencil a 4 vicini (primo ordine, come nella tesi); il **multistencil** verrà aggiunto più avanti.

In [ ]:
%%writefile step1.cu
#include <cstdio>
#include <cmath>
#include <vector>
#include <chrono>
#include <fstream>

// Valore "infinito": nodo non ancora raggiunto dal fronte d'onda.
// Usiamo un numero finito molto grande (non std::numeric_limits<float>::infinity())
// cosi' che a - b tra due INF faccia 0 e non NaN.
const float INF = 1e30f;

// Spostamenti (di, dj) verso i 4 vicini: sinistra, destra, sotto, sopra.
const int DI[4] = { 0, 0, -1, 1 };
const int DJ[4] = { -1, 1, 0, 0 };

// Algoritmo 2 della tesi: soluzione dell'equazione quadratica (3.1)
//   (u - a)^2 + (u - b)^2 = (F h)^2
// a = minimo dei vicini lungo x, b = minimo dei vicini lungo y.
// __host__ __device__: la stessa funzione servira' anche nei kernel GPU.
__host__ __device__ float solve_quadratic(float a, float b, float F, float h)
{
    float Fh = F * h;
    if (fabsf(a - b) <= Fh)      // il fronte arriva da entrambe le direzioni
        return 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));
    return fminf(a, b) + Fh;     // il fronte arriva da una sola direzione
}

// g(x) della tesi: calcola il nuovo valore nel nodo (i, j).
// Legge i 4 vicini (INF se fuori dal dominio) e chiama l'Algoritmo 2.
// Restituisce min(u_vecchio, u_nuovo): u puo' solo diminuire (eq. 3.2).

float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int i, int j, int N, float h)
{
    float left  = (j > 0)     ? u[i * N + (j - 1)] : INF;
    float right = (j < N - 1) ? u[i * N + (j + 1)] : INF;
    float down  = (i > 0)     ? u[(i - 1) * N + j] : INF;
    float up    = (i < N - 1) ? u[(i + 1) * N + j] : INF;

    float a = fminf(left, right);
    float b = fminf(down, up);

    int idx = i * N + j;
    return fminf(u[idx], solve_quadratic(a, b, F[idx], h));
}

// Test dei nodi adiacenti (parte finale dell'Algoritmo 6).
// Per ogni vicino di x che NON e' in A: se g(x_nb) migliora u(x_nb),
// lo aggiorniamo e lo aggiungiamo ad A.

void test_neighbors(int x, std::vector<float>& u, const std::vector<float>& F,
                    int N, float h, int src,
                    std::vector<char>& inA, std::vector<int>& newA)
{
    int i = x / N, j = x % N;
    for (int k = 0; k < 4; ++k) {
        int ni = i + DI[k], nj = j + DJ[k];
        if (ni < 0 || ni >= N || nj < 0 || nj >= N) continue;  // fuori dal dominio
        int nb = ni * N + nj;
        if (nb == src || inA[nb]) continue;                     // sorgente fissa / gia' in A

        float p = u[nb];
        float q = solve_node(u, F, ni, nj, N, h);
        if (p > q) {
            u[nb] = q;
            inA[nb] = 1;
            newA.push_back(nb);
        }
    }
}

// Algoritmo 6 (Fast Iterative Method), versione sequenziale.
// Restituisce il numero di iterazioni del ciclo while.

int fim_cpu(std::vector<float>& u, const std::vector<float>& F,
            int N, float h, int src, float eps)
{
    std::vector<char> inA(N * N, 0);   // inA[x] = 1 se x appartiene ad A
    std::vector<int>  A;               // lista dei nodi attivi

    // Inizializzazione: la sorgente e' gia' "convergente",
    // quindi facciamo il test dei suoi vicini -> A = 4 vicini della sorgente.
    test_neighbors(src, u, F, N, h, src, inA, A);

    int iter = 0;
    while (!A.empty()) {
        ++iter;
        std::vector<int> newA;        // nuova lista attivi
        std::vector<int> converged;   // nodi convergenti in questa iterazione

        // Fase 1: aggiornamento di tutti i nodi di A
        for (int x : A) {
            float p = u[x];
            float q = solve_node(u, F, x / N, x % N, N, h);
            u[x] = q;
            if (fabsf(p - q) < eps) converged.push_back(x);
            else                    newA.push_back(x);    // resta attivo
        }

        // Fase 2: i convergenti escono da A e testano i loro vicini
        for (int x : converged) inA[x] = 0;
        for (int x : converged) test_neighbors(x, u, F, N, h, src, inA, newA);

        A.swap(newA);
    }
    return iter;
}

// Errore relativo, eq. (4.2): ||u - U|| / ||U||, con U = sqrt(x^2 + y^2).
double relative_error(const std::vector<float>& u, int N, float h)
{
    double num = 0.0, den = 0.0;
    for (int i = 0; i < N; ++i)
        for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h;
            double y = -1.0 + i * h;
            double U = std::sqrt(x * x + y * y);
            double d = u[i * N + j] - U;
            num += d * d;
            den += U * U;
        }
    return std::sqrt(num / den);
}

int main()
{
    const float eps = 1e-6f;
    const int sizes[] = { 51, 101, 201, 401 };
    double prev_err = 0.0;

    printf("   N        h      iterazioni   errore rel.   ordine   tempo [ms]\n");
    for (int N : sizes) {
        float h   = 2.0f / (N - 1);                  // dominio [-1, 1]^2
        int   src = (N / 2) * N + (N / 2);           // nodo centrale -> (0, 0)

        std::vector<float> u(N * N, INF);
        std::vector<float> F(N * N, 1.0f);           // mezzo omogeneo
        u[src] = 0.0f;

        auto t0 = std::chrono::high_resolution_clock::now();
        int iter = fim_cpu(u, F, N, h, src, eps);
        auto t1 = std::chrono::high_resolution_clock::now();
        double ms = std::chrono::duration<double, std::milli>(t1 - t0).count();

        double err = relative_error(u, N, h);
        // ordine di convergenza stimato: h si dimezza, quindi log2(e_prec / e)
        if (prev_err > 0.0)
            printf("%5d  %.5f  %8d     %.3e     %5.2f   %9.2f\n", N, h, iter, err, std::log2(prev_err / err), ms);
        else
            printf("%5d  %.5f  %8d     %.3e       -     %9.2f\n", N, h, iter, err, ms);
        prev_err = err;

        if (N == 101) {
            int c = N / 2;
            printf("\n  Valori attorno alla sorgente (in unita' di h), N = 101:\n");
            for (int i = c + 1; i >= c - 1; --i) {
                printf("   ");
                for (int j = c - 1; j <= c + 1; ++j) printf("%7.4f ", u[i * N + j] / h);
                printf("\n");
            }
            printf("  atteso: 1 sugli assi, sqrt(2) = 1.4142 esatto in diagonale\n\n");

            std::ofstream out("step1.bin", std::ios::binary);
            out.write(reinterpret_cast<const char*>(u.data()), N * N * sizeof(float));
        }
    }
    printf("\nSoluzione N = 101 salvata in step1.bin\n");
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 step1.cu -o step1 && ./step1

### Visualizzazione (N = 101)
A sinistra: curve di livello numeriche (continue) sovrapposte a quelle analitiche (tratteggiate), come in fig. 4.1.
A destra: errore |u − U| in ogni nodo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

N = 101
u = np.fromfile('step1.bin', dtype=np.float32).reshape((N, N))   # riga i = asse y, colonna j = asse x
x = np.linspace(-1, 1, N)
X, Y = np.meshgrid(x, x)
U = np.sqrt(X**2 + Y**2)                                         # soluzione analitica (eq. 4.3)

fig, ax = plt.subplots(1, 2, figsize=(13, 5.5))

levels = np.arange(0.2, 1.5, 0.2)
c1 = ax[0].contour(X, Y, u, levels=levels, colors='tab:blue')
ax[0].contour(X, Y, U, levels=levels, colors='k', linestyles='--')
ax[0].clabel(c1, fmt='%.1f', fontsize=8)
ax[0].plot(0, 0, 'ro')
ax[0].set_aspect('equal'); ax[0].set_title('Curve di livello: FIM (blu) vs analitica (tratteggio)')
ax[0].set_xlabel('x [u.a.]'); ax[0].set_ylabel('y [u.a.]')

im = ax[1].imshow(np.abs(u - U), origin='lower', extent=[-1, 1, -1, 1], cmap='viridis')
fig.colorbar(im, ax=ax[1], label='|u − U|')
ax[1].set_title('Errore assoluto')
ax[1].set_xlabel('x [u.a.]'); ax[1].set_ylabel('y [u.a.]')

plt.tight_layout()
plt.show()